# pyaegean Stage D+ — the combined retrain

Retrains the full joint model on **AGDT + Gorman + Pedalion** (~2.4× the training
tokens — the 2024 baseline's data mix), overlap-audited against the UD-Perseus dev/test
folds **and** PROIEL (Gorman's Herodotus files are excluded at source).

**Targets (UD Perseus test):** UAS ≥ 88.2, LAS ≥ 84.0 (the raised 2024 bar), lemma
above Stage D's 87.71 — and on PROIEL, lemma > 90.38 (the shipped hybrid's number,
which Stage D+ must beat on both folds to replace it).

Run top to bottom on a GPU runtime; same decision points as Stage D.

In [ ]:
# 1 — confirm the GPU
import torch
print(torch.cuda.get_device_name(0))
print("bf16:", torch.cuda.is_bf16_supported())

In [ ]:
# 2 — install the package + a fresh clone
!pip install -q "git+https://github.com/ryanpavlicek/pyaegean" torch transformers numpy
!rm -rf pyaegean_repo
!git clone https://github.com/ryanpavlicek/pyaegean.git pyaegean_repo

In [ ]:
# 3 — build the COMBINED dataset (fetches AGDT + UD folds + PROIEL + Gorman + Pedalion)
#     sanity: with_extras true; extras_audit shows kept/excluded per source;
#     train_tokens should be roughly 1.1-1.3M
!python pyaegean_repo/training/build_full_dataset.py --with-extras

In [ ]:
# 4 — train (6 epochs on ~2.4x data ~= Stage D's 15; watch the per-epoch dev lines)
!python pyaegean_repo/training/train_full.py --model bowphs/GreBerta --epochs 6

**5 — decision point.** If `(LAS + lemma)` was still climbing at the last epoch,
uncomment and run cell 5. **Note the best epoch's `lemma(best=<mode>)`** — cell 6
needs it.

In [ ]:
# 5 — OPTIONAL longer re-train
# !python pyaegean_repo/training/train_full.py --model bowphs/GreBerta --epochs 9

In [ ]:
# 6 — headline measurements: set COMPOSE from cell 4's lemma(best=...) mode
COMPOSE = "lookup-first"  # <-- replace with the best dev mode
!python pyaegean_repo/training/eval_full_ud.py \
    --checkpoint pyaegean_repo/training/out/full/model --compose {COMPOSE} \
    --treebank perseus --split test --out pyaegean_repo/training/out/full/ud-perseus-test.json
!python pyaegean_repo/training/eval_full_ud.py \
    --checkpoint pyaegean_repo/training/out/full/model --compose {COMPOSE} \
    --treebank proiel --split test --out pyaegean_repo/training/out/full/ud-proiel-test.json

In [ ]:
# 7 — save everything to Drive BEFORE the runtime recycles
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/pyaegean-stage-dplus"
!cp -r pyaegean_repo/training/out/full "/content/drive/MyDrive/pyaegean-stage-dplus/"
!cp pyaegean_repo/training/data/full-stats.json "/content/drive/MyDrive/pyaegean-stage-dplus/"
!ls -la "/content/drive/MyDrive/pyaegean-stage-dplus/full" "/content/drive/MyDrive/pyaegean-stage-dplus/full/model"

## What to bring back

- `full-stats.json` — the build's extras audit (kept/excluded per source)
- `full/metrics.json` — training history
- `full/ud-perseus-test.json` and `full/ud-proiel-test.json` — **the verdicts**:
  UAS ≥ 88.2 / LAS ≥ 84.0 takes parsing past the freshest published number;
  PROIEL lemma > 90.38 makes this model the package's lemmatizer too

The checkpoint (`full/model/`) stays in Drive — if the verdicts clear, **this** is the
Stage E export artifact.